# Experiment 07: Enumeration Probe

In [ ]:
SMOKE = True

In [ ]:
import sys
import torch
import time
import json
import re
import pandas as pd
import numpy as np
from pathlib import Path
from PIL import Image

EXP_DIR = Path.cwd()
REPO = EXP_DIR
while REPO != REPO.parent and not ((REPO / ".git").exists() or (REPO / "src").is_dir()):
    REPO = REPO.parent

for p in (EXP_DIR / "_models", REPO / "src"):
    if p.is_dir():
        sys.path.insert(0, str(p))

from frame.config import BaselineConfig
from frame.data import load_frame_items, FrameProvider
from enumerate_engine import extract_items
from frame.engine import QwenFrameEngine
from focus.data.data_models import Response, save_items

sys.path.insert(0, str(REPO / "experiments" / "05-bottleneck-audit" / "_models"))
from number_probe import parse_number

# The probe runs on ONE question template. `number` is not homogeneous: 828 of its
# 2094 questions ask about a specific class ("How many Clips…"), which enumerating
# every FO does not answer, and 830 ask for INSTANCES while an unprompted list names
# CLASSES (2 clips = 1 class, 2 instances). This template is the only one where the
# baseline emits a bare integer and the prefill's natural output — a class list — is
# the same quantity. Single variable: the prefill.
PROBE_Q = (
    "How many different foreign object classes appear in this frame? "
    "Please provide a number."
)

cfg = BaselineConfig(
    model_path=Path("/workspace/repo/experiments/02-lora-sft/runs/02_lora_sft_v1/merged/checkpoint-1720"),
    out_dir=EXP_DIR / "runs" / "07_enumeration",
    n_eval=None,
)
cfg.out_dir.mkdir(parents=True, exist_ok=True)
arm_dir = cfg.out_dir / "e1_enumerate"
arm_dir.mkdir(parents=True, exist_ok=True)

all_items = load_frame_items(cfg, splits=("test",))
q_map = pd.DataFrame(
    [{"qID": it.request.qID, "question": it.request.question} for it in all_items]
)

# ---------------------------------------------------------------- e0: baseline
# Re-derived from the rung 02 artifacts. No GPU: the baseline is never re-run.
print("Computing baseline e0_baseline…")
run02_dir = REPO / "experiments" / "02-lora-sft" / "runs" / "02_lora_sft_v1" / "eval_best"

df_e0 = pd.read_csv(run02_dir / "results.csv")
with open(run02_dir / "references.json") as f:
    refs_e0 = pd.DataFrame(json.load(f))
with open(run02_dir / "predictions.json") as f:
    preds_e0 = pd.DataFrame(json.load(f))

m_e0 = df_e0.merge(refs_e0, on="qID").merge(preds_e0, on="qID").merge(q_map, on="qID")

meta_rows = []
for _, row in refs_e0.iterrows():
    # `primary` holds a LEAF capability, never a group. Map it, or valid_buckets
    # matches nothing. Mirrors experiments/05-bottleneck-audit (verified there).
    OBJECT_RECOGNITION_LEAVES = {
        "object_identification", "object_attributes",
        "spatial_localization_camera", "spatial_localization_situs",
    }
    leaf = row["primary"]
    if isinstance(leaf, dict):
        leaf = leaf.get("group", leaf)
    leaf = str(leaf)
    if leaf in OBJECT_RECOGNITION_LEAVES:
        group = "object_recognition"
    elif leaf == "object_aggregation":
        group = "aggregation"
    else:
        group = "temporal_grounding"   # n=1 orphan -> excluded by valid_buckets
    meta_rows.append({
        "qID": row["qID"],
        "capability_group": group,
        "distribution": "OOD" if row["qID"].startswith("heico") else "ID",
    })
m_e0 = m_e0.merge(pd.DataFrame(meta_rows), on="qID")

e0_data = {"arm": "e0_baseline"}

valid_buckets = ["object_recognition", "aggregation"]
bucket_cols = [
    m_e0[(m_e0["capability_group"] == grp) & (m_e0["distribution"] == dist)]["correctness"].mean()
    for grp in valid_buckets
    for dist in ("ID", "OOD")
]
e0_data["bucket_mean"] = sum(bucket_cols) / len(bucket_cols)

probe_e0 = m_e0[m_e0["question"] == PROBE_Q].copy()
probe_e0["true_count"] = pd.to_numeric(probe_e0["answer"], errors="coerce")
probe_e0["counted"] = probe_e0["content"].apply(parse_number)

e0_data["n_probe"] = len(probe_e0)
e0_data["acc_probe"] = probe_e0["correctness"].mean()
for k in range(1, 5):
    sel = probe_e0[probe_e0["true_count"] == k]
    e0_data[f"n_at_{k}"] = len(sel)
    e0_data[f"acc_at_{k}"] = sel["correctness"].mean() if len(sel) else np.nan
    # What the model SAYS when asked to count. The prefill arm's items@k is read
    # against this, and against nothing else.
    e0_data[f"counted_mean_at_{k}"] = sel["counted"].mean() if len(sel) else np.nan

e0_data["lat_p50_s"] = df_e0["latency"].median()
e0_data["lat_p99_s"] = df_e0["latency"].quantile(0.99)
e0_data["n_timed_out"] = df_e0["timed_out"].sum()

# G3 — the baseline artifacts are the ones we think they are. A failure here is a
# FINDING: never move an expected value to make it pass.
assert abs(e0_data["bucket_mean"] - 0.5503) <= 0.005, f"G3 FAILED bucket_mean: {e0_data['bucket_mean']}"
assert e0_data["n_probe"] == 436, f"G3 FAILED n_probe: {e0_data['n_probe']}"
assert abs(e0_data["acc_at_2"] - 0.4000) <= 0.005, f"G3 FAILED acc@2: {e0_data['acc_at_2']}"
assert abs(e0_data["counted_mean_at_1"] - 1.158) <= 0.02, f"G3 FAILED counted@1: {e0_data['counted_mean_at_1']}"
assert abs(e0_data["counted_mean_at_2"] - 1.400) <= 0.02, f"G3 FAILED counted@2: {e0_data['counted_mean_at_2']}"
print(f"OK G3: baseline reproduces — n={e0_data['n_probe']}, "
      f"counted@1={e0_data['counted_mean_at_1']:.3f}, counted@2={e0_data['counted_mean_at_2']:.3f}")

# --------------------------------------------------------------- e1: the probe
eval_items = [it for it in all_items if it.request.question == PROBE_Q]
assert len(eval_items) == 436, f"G0 FAILED: probe subset is {len(eval_items)}, expected 436"

if SMOKE:
    # One item per video, strided, so the smoke spans videos instead of one clip.
    by_video = {}
    for it in eval_items:
        by_video.setdefault(it.video_id, it)
    videos = sorted(by_video)
    stride = max(1, len(videos) // 20)
    eval_items = [by_video[v] for v in videos[::stride]][:20]
    assert len({it.video_id for it in eval_items}) == len(eval_items), "G0 FAILED: duplicate video in smoke"
    print(f"SMOKE: {len(eval_items)} items over {len(eval_items)} distinct videos "
          f"(of {len(videos)} with this question)")


class EnumerateFrameEngine(QwenFrameEngine):
    """Production engine, one change: the assistant turn is prefilled with `ITEMS:`.

    The template ends at `<|im_start|>assistant\\n`; appending `ITEMS:` starts the
    sentence for the model instead of asking it to. v1 asked via SYSTEM_PROMPT and
    was ignored 20/20 — 13.7k training examples of a bare answer outweigh a request.
    """

    @torch.no_grad()
    def predict(self, image, question: str) -> str:
        from qwen_vl_utils import process_vision_info
        try:
            messages = self._messages(image, question)
            text = self.processor.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True
            )
            text += "ITEMS:"

            image_inputs, video_inputs = process_vision_info(messages)
            inputs = self.processor(
                text=[text],
                images=image_inputs,
                videos=video_inputs,
                padding=True,
                return_tensors="pt",
            ).to(self.model.device)

            gen_ids = self.model.generate(
                **inputs, max_new_tokens=self._cfg.max_new_tokens, do_sample=False
            )
            trimmed = gen_ids[0][inputs.input_ids.shape[1]:]
            out = self.processor.decode(trimmed, skip_special_tokens=True).strip()
        except Exception as exc:
            import logging
            logging.getLogger(__name__).error("Inference failed: %s", exc)
            return f"Inference Error: {str(exc)[:60]}"

        return out[: self._cfg.answer_char_cap]


engine = EnumerateFrameEngine(cfg)
engine.load()

responses, raw_contents, raw_tokens = [], {}, {}

frames_dir = Path("/workspace/frames_cache")
if not frames_dir.exists():
    frames_dir = REPO / "experiments/05-bottleneck-audit/runs/05_bottleneck_audit/frames"

print(f"Starting inference… SMOKE={SMOKE}, n={len(eval_items)}")
for item in eval_items:
    image_path = frames_dir / f"{item.request.qID}.png"
    if image_path.exists():
        image = Image.open(image_path).copy()
    else:
        provider = FrameProvider(cfg)
        provider.ensure_reader(item)
        image = provider.get_frame(item)
        provider.close()

    t_start = time.perf_counter()
    raw_content = engine.predict(image, item.request.question)
    latency = time.perf_counter() - t_start

    raw_tokens[item.request.qID] = len(engine.processor.tokenizer(raw_content).input_ids)
    raw_contents[item.request.qID] = raw_content
    responses.append(Response(qID=item.request.qID, content=raw_content, latency=latency))

    if SMOKE:
        base = preds_e0[preds_e0["qID"] == item.request.qID]
        base_txt = base.iloc[0]["content"] if not base.empty else "?"
        truth = next(i.reference.answer for i in all_items if i.request.qID == item.request.qID)
        print(f"\n--- {item.request.qID} | truth={truth} | baseline='{base_txt}' ---")
        print(f"    prefill -> {raw_content!r}   items={extract_items(raw_content)}")

engine.unload()

save_items(responses, arm_dir / "predictions.json")
with open(arm_dir / "predictions_raw.json", "w") as f:
    json.dump(raw_contents, f, indent=2)

# ------------------------------------------------------------------- G1 and G2
# Both run in SMOKE too. In v1 G1 sat inside `if not SMOKE:`, so a run that failed
# 20/20 still printed "SMOKE completed cleanly".

# G1 — did the prefill take? A bare integer means the model answered the question
# as usual and the prefill changed nothing. This, not a diff against the baseline,
# is the real check: for this template any enumeration necessarily differs.
BARE_NUMBER = re.compile(r"^\s*\d+\s*\.?\s*$")
n_bare = sum(1 for t in raw_contents.values() if BARE_NUMBER.match(t))
bare_frac = n_bare / len(raw_contents)
print(f"\nG1: {n_bare}/{len(raw_contents)} outputs are still a bare integer ({bare_frac:.1%})")
assert bare_frac <= 0.10, f"G1 FAILED: prefill did not take on {bare_frac:.1%} of outputs"

truth_map = {it.request.qID: it.reference.answer for it in all_items}
e1 = pd.DataFrame([
    {
        "qID": q,
        "raw_content": t,
        "items": extract_items(t),
        "true_count": pd.to_numeric(truth_map.get(q), errors="coerce"),
        "n_tokens": raw_tokens[q],
        "latency": r.latency,
    }
    for (q, t), r in zip(raw_contents.items(), responses)
])

e1_data = {"arm": "e1_enumerate", "n_probe": len(e1)}
for k in range(1, 5):
    sel = e1[e1["true_count"] == k]
    e1_data[f"n_at_{k}"] = len(sel)
    e1_data[f"items_mean_at_{k}"] = sel["items"].mean() if len(sel) else np.nan
    e1_data[f"items_std_at_{k}"] = sel["items"].std() if len(sel) else np.nan
if len(e1) > 1:
    e1_data["spearman_rho"] = e1["true_count"].corr(e1["items"], method="spearman")
e1_data["bare_frac"] = bare_frac
e1_data["lat_p50_s"] = e1["latency"].median()
e1_data["lat_p99_s"] = e1["latency"].quantile(0.99)
e1_data["max_tokens_seen"] = e1["n_tokens"].max()

# G2 — the control. If the model lists ~2 things when there is 1, it confabulates
# and "lists 2 when there are 2" proves nothing. READ THIS BEFORE items@2.
print(f"G2: items@1 = {e1_data['items_mean_at_1']:.3f} (n={e1_data['n_at_1']}) "
      f"vs baseline counted@1 = {e0_data['counted_mean_at_1']:.3f}")
print(f"    items@2 = {e1_data['items_mean_at_2']:.3f} (n={e1_data['n_at_2']}) "
      f"vs baseline counted@2 = {e0_data['counted_mean_at_2']:.3f}")

if not SMOKE:
    assert len(e1) == 436, f"G4 FAILED: n is {len(e1)}"
    print("OK G4: n == 436")
    pd.DataFrame([e0_data, e1_data]).to_csv(EXP_DIR / "RESULTS_enumeration.csv", index=False)
    e1.to_csv(arm_dir / "inspect.csv", index=False)
    print("Wrote RESULTS_enumeration.csv")
else:
    print("\nSMOKE done. Numbers above are n<=20 — NOT the result. The rule reads the full.")
